# Compare TCIA downloaders on the same images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kirbyju/TCIA_Notebooks/blob/main/benchmarks/TCIA_Downloader_Comparison.ipynb)

Let's check how the pip-installed TCIA Data Retriever and IDC tools perform on the **same public [CMB-GEC](https://www.cancerimagingarchive.net/collection/cmb-gec/) images**. We use two CT/MR series containing 96 DICOM instances, about 29 MB per download. The selected images are licensed CC BY 4.0; consult the collection page for the dataset citation. This is a small pilot, so it cannot tell us which tool is fastest for every collection or file type.

The benchmark alternates the tools, uses a new destination each time, and checks that every DICOM instance is present and **byte-for-byte identical**. We compare both fresh-process startup and a prepared-client case, since an existing analysis session may already have loaded imaging metadata. Installation and verification are excluded from timing.

There are three trials per tool in each case: about **348 MB total payload** with these defaults. Public cloud caches cannot be reset by this notebook. Network conditions and the tools' default concurrency can affect the result. No controlled data or credentials are used.

Read the [TCIA data usage policy](https://www.cancerimagingarchive.net/data-usage-policies-and-restrictions/) before running the transfer cells. Those cells explicitly pass `--accept-data-policy`. Running them records your acceptance for these downloads.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "tcia-data-retriever==0.1.0rc1", "idc-index==0.13.4", "idc-index-data==25.0.5", "pydicom"])


## Set up the bounded comparison

The cohort and runner are embedded below, so you do not need to clone a repository or download executable code from a moving branch. Expand the cell to inspect them.

The selected cloud object prefixes match both packages' metadata. The runner checks the IDC data version and source URLs again before comparing results. It stops if a tool fails or returns a different set of images.

In [ ]:
from pathlib import Path
import json

RUNNER_SOURCE = '"""Bounded public-data comparison of the released TDR and IDC Python interfaces.\n\nRun this module as a script. Every trial runs in a child process with a fresh\noutput directory; installation and post-download verification are not timed.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom datetime import datetime, timezone\nimport hashlib\nfrom importlib.metadata import version\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport signal\nimport statistics\nimport subprocess\nimport sys\nimport threading\nfrom time import perf_counter, sleep\n\nPOLICY = "https://www.cancerimagingarchive.net/data-usage-policies-and-restrictions/"\n\n\ndef inventory(directory, cohort):\n    import pydicom\n    expected = {row[\'SeriesInstanceUID\']: int(row[\'instanceCount\']) for row in cohort[\'series\']}\n    counts = {uid: 0 for uid in expected}\n    objects = {}\n    size = 0\n    for path in directory.rglob(\'*.dcm\'):\n        ds = pydicom.dcmread(path, stop_before_pixels=True,\n                            specific_tags=[\'SeriesInstanceUID\', \'SOPInstanceUID\'])\n        uid, sop = str(ds.SeriesInstanceUID), str(ds.SOPInstanceUID)\n        if uid not in counts or sop in objects:\n            raise RuntimeError(\'Unexpected series or duplicate DICOM instance\')\n        payload = path.read_bytes()\n        objects[sop] = hashlib.sha256(payload).hexdigest()\n        size += len(payload)\n        counts[uid] += 1\n    if counts != expected:\n        raise RuntimeError(f\'DICOM instance counts differ: {counts} vs {expected}\')\n    fingerprint = hashlib.sha256(json.dumps(objects, sort_keys=True).encode()).hexdigest()\n    return {\'instances\': len(objects), \'payload_bytes\': size, \'fingerprint\': fingerprint,\n            \'series_instance_counts\': counts}\n\n\ndef trial(engine, cohort_path, destination, result_path, warm=False):\n    cohort = json.loads(cohort_path.read_text())\n    if destination.exists():\n        raise RuntimeError(\'A trial must have a new output directory\')\n    destination.mkdir(parents=True)\n    first = []\n    stop = threading.Event()\n    started = perf_counter()\n\n    def watch():\n        while not stop.is_set():\n            for path in destination.rglob(\'*.dcm\'):\n                try:\n                    if path.stat().st_size:\n                        first.append(perf_counter() - started)\n                        return\n                except FileNotFoundError:\n                    pass\n            stop.wait(0.1)\n\n    watcher = threading.Thread(target=watch, daemon=True)\n    watcher.start()\n    validation_seconds = 0\n    initialization_seconds = 0\n    try:\n        engine_report = None\n        if engine == \'tdr\':\n            import tcia_data_retriever as tdr\n            manifest = destination.parent / (destination.name + \'.s5cmd\')\n            manifest.write_text(\'\'.join(\'cp \'+r[\'current_url\']+\' ./\\n\' for r in cohort[\'series\']))\n            initialization_seconds = perf_counter() - started\n            if warm:\n                started = perf_counter()\n            engine_report = tdr.download(manifest, destination, accept_data_policy=True,\n                                         skip_existing=False, max_retries=0)\n            if engine_report[\'total\'] != len(cohort[\'series\']):\n                raise RuntimeError(\'TDR did not account for all selected series\')\n        else:\n            from idc_index import IDCClient\n            client = IDCClient()\n            if client.get_idc_version() != cohort[\'idc_version\']:\n                raise RuntimeError(\'IDC data version differs from the pinned cohort\')\n            initialization_seconds = perf_counter() - started\n            checking = perf_counter()\n            index = client.index.set_index(\'SeriesInstanceUID\')\n            for row in cohort[\'series\']:\n                source = index.loc[row[\'SeriesInstanceUID\']]\n                actual = \'s3://\' + source.aws_bucket + \'/\' + source.crdc_series_uuid + \'/*\'\n                if actual != row[\'current_url\']:\n                    raise RuntimeError(\'The two engines would use different source objects\')\n            validation_seconds = perf_counter() - checking\n            if warm:\n                original = client._s5cmd_run\n                client._s5cmd_run = lambda **kw: None\n                try:\n                    client.download_dicom_series(\n                        seriesInstanceUID=[cohort[\'series\'][0][\'SeriesInstanceUID\']],\n                        downloadDir=str(destination.parent / (\'.metadata-warmup-\'+destination.name)),\n                        use_s5cmd_sync=False, quiet=True, show_progress_bar=False)\n                finally:\n                    client._s5cmd_run = original\n                started = perf_counter()\n                validation_seconds = 0\n            client.download_dicom_series(\n                seriesInstanceUID=[r[\'SeriesInstanceUID\'] for r in cohort[\'series\']],\n                downloadDir=str(destination), use_s5cmd_sync=False,\n                quiet=True, show_progress_bar=False)\n        elapsed = perf_counter() - started - validation_seconds\n    finally:\n        stop.set()\n        watcher.join(timeout=2)\n    checked = inventory(destination, cohort)\n    result = {\'engine\': engine, \'ok\': True, \'elapsed_seconds\': elapsed,\n              \'first_dicom_visible_seconds\': first[0] - validation_seconds if first else None,\n              \'initialization_seconds\': initialization_seconds, \'source_validation_seconds_excluded\': validation_seconds,\n              \'warm_prepared_client\': warm,\n              \'MiB_per_second\': checked[\'payload_bytes\'] / 2**20 / elapsed,\n              \'engine_report\': engine_report, **checked}\n    result_path.write_text(json.dumps(result, indent=2)+\'\\n\')\n\n\ndef run(cohort_path, output, repeats=3, max_total_mb=1000, timeout=180, warm=False):\n    cohort = json.loads(cohort_path.read_text())\n    planned = sum(float(r[\'series_size_MB\']) * 1_000_000 for r in cohort[\'series\']) * 2 * repeats\n    if repeats < 1 or planned <= 0 or planned * 1.1 > max_total_mb * 1_000_000:\n        raise ValueError(\'Selection exceeds the configured total payload budget\')\n    if output.exists():\n        raise ValueError(\'Use a fresh benchmark output directory\')\n    output.mkdir(parents=True)\n    results = []\n    report = {\'observed_at_utc\': datetime.now(timezone.utc).isoformat(),\n              \'platform\': platform.platform(), \'python\': platform.python_version(),\n              \'versions\': {p: version(p) for p in [\'tcia-data-retriever\', \'idc-index\', \'idc-index-data\', \'pydicom\']},\n              \'cohort\': cohort, \'cohort_sha256\': hashlib.sha256(cohort_path.read_bytes()).hexdigest(),\n              \'expected_total_payload_bytes\': int(planned), \'max_total_MB\': max_total_mb,\n              \'mode\': \'warm prepared-client API calls\' if warm else \'fresh-process end-to-end\',\n              \'destination_mode\': \'fresh destination, normal copy, same S3 objects; alternating order\',\n              \'settings\': {\'tdr_processes\': 8, \'tdr_max_retries\': 0,\n                           \'idc_workers\': \'package default\', \'idc_retries\': \'package default\',\n                           \'idc_sync\': False}, \'trials\': results}\n    import tcia_data_retriever as tdr\n    report[\'tdr_build\'] = tdr.build_info()\n    script = Path(__file__).resolve()\n    for repeat in range(repeats):\n        order = [\'tdr\', \'idc\'] if repeat % 2 == 0 else [\'idc\', \'tdr\']\n        for engine in order:\n            name = f\'{repeat+1:02d}-{engine}\'\n            destination, result_path = output / name, output / (name+\'.json\')\n            args = [sys.executable, str(script), \'--trial\', engine, \'--cohort\', str(cohort_path),\n                    \'--output\', str(destination), \'--result\', str(result_path), \'--accept-data-policy\']\n            if warm:\n                args.append(\'--warm\')\n            print(f\'Running {name}: {len(cohort["series"])} series in a fresh directory\', flush=True)\n            with (output / (name+\'.log\')).open(\'w\') as log:\n                process = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT,\n                                           start_new_session=True)\n                try:\n                    code = process.wait(timeout=timeout)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGINT)\n                    try:\n                        process.wait(timeout=10)\n                    except subprocess.TimeoutExpired:\n                        os.killpg(process.pid, signal.SIGKILL)\n                        process.wait()\n                    code = 124\n            if code or not result_path.is_file():\n                results.append({\'engine\': engine, \'repeat\': repeat+1, \'ok\': False,\n                                \'returncode\': code, \'log\': name+\'.log\'})\n                report[\'status\'] = \'failed; no speed conclusion; stopped after first failure\'\n                (output/\'report.json\').write_text(json.dumps(report, indent=2)+\'\\n\')\n                raise RuntimeError(f\'{name} failed; inspect {output / (name+".log")}\')\n            item = json.loads(result_path.read_text()); item[\'repeat\'] = repeat+1\n            results.append(item)\n            if sum(r.get(\'payload_bytes\', 0) for r in results) > max_total_mb*1_000_000:\n                raise RuntimeError(\'Observed payload budget exceeded\')\n            if len({r[\'fingerprint\'] for r in results}) != 1:\n                raise RuntimeError(\'Downloaded DICOM bytes differ; do not compare performance as equivalent data\')\n            (output/\'report.json\').write_text(json.dumps(report, indent=2)+\'\\n\')\n    report[\'status\'] = \'success\'\n    report[\'all_payloads_identical\'] = True\n    report[\'summary\'] = []\n    for engine in [\'tdr\', \'idc\']:\n        rows = [r for r in results if r[\'engine\'] == engine]\n        report[\'summary\'].append({\'engine\': engine, \'trials\': len(rows),\n            \'median_seconds\': statistics.median(r[\'elapsed_seconds\'] for r in rows),\n            \'median_MiB_per_second\': statistics.median(r[\'MiB_per_second\'] for r in rows)})\n    report[\'limitations\'] = [\'Small CT/MR pilot; no large-file or controlled-access speed claim.\',\n        \'Fresh local destinations do not clear remote/CDN caches.\',\n        \'Default concurrency differs; TDR retries disabled; IDC uses package defaults.\',\n        \'Cold elapsed time includes imports, index/preparation and transfer; warm mode excludes client initialization.\',\n        \'Warm IDC uses a metadata-only preparation call with transfer intercepted; TDR always starts its packaged executable per call.\',\n        \'Installation, source validation and post-download verification are not timed.\',\n        \'First visible DICOM is a filesystem observation, not a network first-byte measurement.\']\n    (output/\'report.json\').write_text(json.dumps(report, indent=2)+\'\\n\')\n    print(json.dumps(report[\'summary\'], indent=2), flush=True)\n    return report\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--cohort\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--repeats\', type=int, default=3)\n    p.add_argument(\'--max-total-mb\', type=float, default=1000)\n    p.add_argument(\'--timeout\', type=float, default=180)\n    p.add_argument(\'--accept-data-policy\', action=\'store_true\')\n    p.add_argument(\'--warm\', action=\'store_true\')\n    p.add_argument(\'--trial\', choices=[\'tdr\', \'idc\'])\n    p.add_argument(\'--result\', type=Path)\n    args = p.parse_args()\n    if not args.accept_data_policy:\n        p.error(\'Explicit policy acceptance is required: \'+POLICY)\n    if args.trial:\n        trial(args.trial, args.cohort.resolve(), args.output.resolve(), args.result.resolve(), args.warm)\n    else:\n        run(args.cohort.resolve(), args.output.resolve(), args.repeats, args.max_total_mb, args.timeout, args.warm)\n\n\nif __name__ == \'__main__\':\n    main()\n'
COHORT = {'idc_version': 'v25', 'collection': 'CMB-GEC', 'tdr_source_commit': '618d8bc4e493cabc20b9afc41d54a3ce8a2b0df5', 'source_manifest_sha256': 'c49654b1c9e2f24e63da15ecc8279e83c42b0790e3b3460d65805e9d20f45a86', 'series': [{'SeriesInstanceUID': '1.3.6.1.4.1.14519.5.2.1.290156256655956441687973099292341977617', 'StudyInstanceUID': '1.3.6.1.4.1.14519.5.2.1.202140194759751273168695877937723717335', 'PatientID': 'MSB-04437', 'Modality': 'CT', 'series_size_MB': 16.924496, 'instanceCount': 32, 'current_url': 's3://idc-open-data/0d904629-788e-488a-a6d7-330984c94aaf/*', 'license_short_name': 'CC BY 4.0'}, {'SeriesInstanceUID': '1.3.6.1.4.1.14519.5.2.1.211954693908121002876720024519773672519', 'StudyInstanceUID': '1.3.6.1.4.1.14519.5.2.1.245380081716850000484974983856441313026', 'PatientID': 'MSB-04437', 'Modality': 'MR', 'series_size_MB': 12.088898, 'instanceCount': 64, 'current_url': 's3://idc-open-data/5f770d9d-57f7-43a6-8e3c-5d707035c269/*', 'license_short_name': 'CC BY 4.0'}]}
ROOT = Path("tcia-downloader-benchmark")
ROOT.mkdir(exist_ok=True)
RUNNER = ROOT / "benchmark_tcia_downloads.py"
COHORT_PATH = ROOT / "cohort.json"
RUNNER.write_text(RUNNER_SOURCE)
COHORT_PATH.write_text(json.dumps(COHORT, indent=2))
print("Selected series:", len(COHORT["series"]))
print("DICOM instances:", sum(r["instanceCount"] for r in COHORT["series"]))


In [ ]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

REPEATS = 3
TOTAL_BUDGET_MB = 1000
expected_mb = sum(r["series_size_MB"] for r in COHORT["series"]) * REPEATS * 4
assert 0 < expected_mb * 1.1 < TOTAL_BUDGET_MB, "Reduce the cohort or repeat count."
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"Expected total payload for both cases: {expected_mb:.1f} MB")


## Run both cases

**Fresh-process** includes package imports, client/index initialization and the transfer call. **Prepared-client** prepares IDC's index and download query first, without downloading images. The Data Retriever's public Python API starts its packaged executable for every call, so that executable's setup remains part of its call time.

These cases answer different questions. A faster fresh start does not by itself prove faster network transfer. The tools also have different default concurrency settings. This benchmark measures their Python interfaces as used here; it does not claim a controlled comparison of networking algorithms.

Each trial writes a log and checks DICOM identifiers, counts and SHA-256 hashes after timing. Fresh destinations prevent previously downloaded files from silently turning into a no-transfer test.

In [ ]:
reports = []
for mode in ["cold", "warm"]:
    output = ROOT / f"{RUN_ID}-{mode}"
    command = [sys.executable, str(RUNNER), "--cohort", str(COHORT_PATH),
               "--output", str(output), "--repeats", str(REPEATS),
               "--max-total-mb", str(TOTAL_BUDGET_MB), "--accept-data-policy"]
    if mode == "warm":
        command.append("--warm")
    subprocess.run(command, check=True)
    report = json.loads((output / "report.json").read_text())
    reports.append(report)
    assert report["all_payloads_identical"]
    print(mode, "passed content verification; report:", output / "report.json")


## Compare the results

Lower elapsed time is better. Compare the two cases separately and keep the raw trial timings: a single very fast or slow run can move a three-trial median. The first-visible-file measurement is a filesystem observation, not a network first-byte measurement.

Keep the JSON reports when sharing results. They record package versions, source revisions, the selected data, settings and limitations. We still need larger and different file shapes before making broad speed claims.

In [ ]:
summary = pd.DataFrame([
    {"case": report["mode"], **row}
    for report in reports for row in report["summary"]
])
display(summary)
trials = pd.DataFrame([
    {"case": report["mode"], **{k: row[k] for k in
        ["engine", "repeat", "elapsed_seconds", "MiB_per_second", "instances", "payload_bytes"]}}
    for report in reports for row in report["trials"]
])
display(trials)
print("All files matched across trials:", len({r["fingerprint"] for report in reports for r in report["trials"]}) == 1)
